# QM9 3D Geometry Utilities Demo

Demonstrates `miniqm.geometry` (`zero_center_of_mass`, `random_rotation`, and `kabsch_align`) on real QM9 molecules from the validation split.

In [ ]:
import pathlib
from typing import Final

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from miniqm import data_source, geometry

DATA_DIR: Final = pathlib.Path("../data").resolve()
VALIDATION_PATH: Final = DATA_DIR / "validation.parquet"

source = data_source.DataSource(VALIDATION_PATH, max_molecules=64)
sample_batch = next(source.iter_batches(batch_size=16))

positions = jnp.asarray(sample_batch.positions)
atom_mask = jnp.asarray(sample_batch.atom_mask)
print(f"positions: {positions.shape}, atom_mask: {atom_mask.shape}")

In [ ]:
shifted = positions + jnp.array([5.0, -3.0, 2.0], dtype=jnp.float32)
recentered = geometry.zero_center_of_mass(shifted, atom_mask)

max_com_err = float(jnp.max(jnp.abs(recentered - positions)))
max_pad_norm = float(jnp.max(jnp.abs(recentered * (~atom_mask)[..., None])))
print(f"Max reconstruction error after shift + zero_center_of_mass: {max_com_err:.2e} A")
print(f"Max padded slot magnitude: {max_pad_norm:.2e} A")

In [ ]:
rotated = geometry.random_rotation(jax.random.key(0), positions, atom_mask)
translated_rotated = (rotated + jnp.array([3.0, -4.0, 1.5], dtype=jnp.float32)) * atom_mask[
    ..., None
]
aligned = geometry.kabsch_align(
    reference=positions,
    target=translated_rotated,
    atom_mask=atom_mask,
)

pair_mask = atom_mask[:, :, None] & atom_mask[:, None, :]
orig_dists = jnp.linalg.norm(positions[:, :, None, :] - positions[:, None, :, :], axis=-1)
rot_dists = jnp.linalg.norm(rotated[:, :, None, :] - rotated[:, None, :, :], axis=-1)
max_dist_diff = float(jnp.max(jnp.abs(orig_dists - rot_dists) * pair_mask))
max_align_err = float(jnp.max(jnp.abs(aligned - positions)))

print(f"Max pairwise distance drift under random_rotation: {max_dist_diff:.2e} A")
print(f"Max Kabsch alignment error on rotated + translated batch: {max_align_err:.2e} A")

In [ ]:
reflected = positions * jnp.array([1.0, 1.0, -1.0], dtype=jnp.float32)
aligned_reflected = geometry.kabsch_align(
    reference=positions,
    target=reflected,
    atom_mask=atom_mask,
)

atom_counts = jnp.sum(atom_mask, axis=-1)
proper_rmsd = jnp.sqrt(jnp.sum((aligned - positions) ** 2, axis=(-1, -2)) / atom_counts)
reflected_rmsd = jnp.sqrt(
    jnp.sum((aligned_reflected - positions) ** 2, axis=(-1, -2)) / atom_counts
)

mol_idx = int(jnp.argmax(reflected_rmsd))
num_valid = int(atom_counts[mol_idx])
mol_bonds = sample_batch.bond_features[mol_idx, :num_valid, :num_valid]
bonded_pairs = np.argwhere(mol_bonds[:, :, 0] == 0.0)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
views = (
    (
        "Reference vs. Kabsch-Aligned Rotated",
        positions[mol_idx],
        aligned[mol_idx],
        proper_rmsd[mol_idx],
    ),
    ("Reference vs. Unaligned Rotated", positions[mol_idx], rotated[mol_idx], float("nan")),
    (
        "Reference vs. Kabsch-Aligned Mirror Enantiomer",
        positions[mol_idx],
        aligned_reflected[mol_idx],
        reflected_rmsd[mol_idx],
    ),
)

for ax, (title, ref_xyz, cmp_xyz, rmsd_val) in zip(axes, views, strict=True):
    ref_np = np.asarray(ref_xyz[:num_valid])
    cmp_np = np.asarray(cmp_xyz[:num_valid])
    for src_atom, dst_atom in bonded_pairs:
        if src_atom < dst_atom:
            ax.plot(
                [ref_np[src_atom, 0], ref_np[dst_atom, 0]],
                [ref_np[src_atom, 1], ref_np[dst_atom, 1]],
                color="tab:blue",
                alpha=0.5,
            )
            ax.plot(
                [cmp_np[src_atom, 0], cmp_np[dst_atom, 0]],
                [cmp_np[src_atom, 1], cmp_np[dst_atom, 1]],
                color="tab:orange",
                linestyle="--",
                alpha=0.6,
            )
    ax.scatter(ref_np[:, 0], ref_np[:, 1], color="tab:blue", s=50, label="Reference")
    ax.scatter(cmp_np[:, 0], cmp_np[:, 1], color="tab:orange", marker="x", s=60, label="Target")
    subtitle = f"\n(RMSD = {float(rmsd_val):.3f} A)" if not np.isnan(float(rmsd_val)) else ""
    ax.set_title(f"{title}{subtitle}", fontsize=10)
    ax.set_xlabel("x (A)")
    ax.set_ylabel("y (A)")
    ax.set_aspect("equal")
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()